# WAKIS: Rechteckresonator mit einer SIBC-Wand

**Ziel:** PEC-Referenzfrequenz des $\mathrm{TE}_{101}$-Modes und erste quantitative Validierung der SIBC-Wandverluste. Fünf Wände sind äußere PEC-Randbedingungen; die sechste Wand (bei $x=x_w$) ist die ebene Grenzfläche eines STL-Blocks. Für beide Varianten wird **dieselbe STL-Geometrie** genutzt, mit Materialtyp `pec` bzw. `sibc`.

Mit `conformal_pec_snap_fraction=0.2` werden kleine aktive Schnittanteile an der Leitergrenze gesnappt. **PEC und SIBC verwenden dieselbe Snap-Fraction.** Die Wand ist bewusst leicht gegenüber dem kartesischen Gitter versetzt, damit echte konforme Schnittflächen entstehen. Die geometrisch aktive Kavität hat $a=L_x-x_w$, $b=L_y$, $d=L_z$. Die äußere x-Min-Randbedingung liegt *hinter* dem Leiter. Für die erste Validierung wird die Eigenmode mit der analytischen TE101-Frequenz verglichen und der Wandverlust über eine Störung erster Ordnung ermittelt. Das ist zuverlässiger als ein grober Frequenzsweep bei hohem Q.

**Voraussetzungen:** Aktuelle `SolverFIT3D`-Materialkorrektur und `SolverFIT3D_FD` mit konsequentem `order="F"` sowie aktiver $D_{Z_s}$-Korrektur. Das Notebook wurde nicht in deiner vollständigen WAKIS-Laufzeit ausgeführt.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import numpy as np
import pyvista as pv
from scipy.constants import c, mu_0, pi
from wakis import GridFIT3D
from wakis.solverFIT3D_FD import SolverFIT3D_FD

# Geometry: all values in meters
Lx, Ly, Lz = 0.36, 0.22, 0.26
x_wall = 0.087  # not on a mesh plane (dx = 0.02 m)
Nx, Ny, Nz = 18, 11, 13
eta = 0.2  # PEC/SIBC conformal snapping threshold
sigma_wall = 5.8e7  # copper-like conductivity [S/m]

# Inner rectangular vacuum cavity: x_wall < x < Lx
# TE101: E_y ~ sin(pi*(x-x_wall)/a) sin(pi*z/d)
a, b, d = Lx - x_wall, Ly, Lz
kx, kz = pi / a, pi / d
f_analytical = c / 2 * np.sqrt((1 / a) ** 2 + (1 / d) ** 2)
print(f"TE101 analytical PEC frequency: {f_analytical / 1e9:.6f} GHz")

In [ ]:
tmp = TemporaryDirectory()
stl_path = Path(tmp.name) / "left_wall.stl"
# A single closed box extends beyond the x-low, y and z simulation boundaries.
# Inside the domain, only its right face x=x_wall is physically exposed.
solid = pv.Cube(
    center=((x_wall - 0.12) / 2, Ly / 2, Lz / 2),
    x_length=x_wall + 0.12,
    y_length=Ly + 0.20,
    z_length=Lz + 0.20,
).triangulate()
solid.save(str(stl_path))


def make_solver(material_type):
    mat = [np.inf, 1.0, 0.0] if material_type == "pec" else [1.0, 1.0, sigma_wall]
    grid = GridFIT3D(
        xmin=0.0,
        xmax=Lx,
        ymin=0.0,
        ymax=Ly,
        zmin=0.0,
        zmax=Lz,
        Nx=Nx,
        Ny=Ny,
        Nz=Nz,
        stl_solids={"wall": str(stl_path)},
        stl_materials={"wall": mat},
        stl_material_types={"wall": material_type},
        geometry_mode="conformal",
        stl_method="interior_points",
        bg_material_type="normal",
        conformal_pec_snap_fraction=eta,
        verbose=0,
    )
    return SolverFIT3D_FD(
        grid=grid,
        use_stl=True,
        use_sibc=False,
        bc_low=["PEC"] * 3,
        bc_high=["PEC"] * 3,
        verbose=0,
    )


pec = make_solver("pec")
sibc = make_solver("sibc")
print("E DOFs (PEC/SIBC):", len(pec.free_E_idx), len(sibc.free_E_idx))
assert np.array_equal(pec.free_E_idx, sibc.free_E_idx), "PEC/SIBC DOF mismatch"
print("SIBC face coefficients:", np.count_nonzero(sibc.sibc_g > 0))
assert sibc.has_sibc

# Verify that the same snapping produces comparable reduced PEC/SIBC operators.
assert pec.C_red.shape == sibc.C_red.shape
assert np.allclose(pec.Meps.diagonal(), sibc.Meps.diagonal()), "PEC/SIBC Meps differ"
assert np.allclose(pec.Mnu.diagonal(), sibc.Mnu.diagonal()), "PEC/SIBC Mnu differ"
print(f"Conformal snap fraction: {eta:.2f}")

## 1. Geometrische und algebraische Kontrolle

Die SIBC-Flächen sollen bei diesem Aufbau überwiegend y- und z-normal sein, weil die ebene Leiteroberfläche senkrecht zur x-Achse liegt und entsprechende FIT-Flächen schneidet. Prüfe, ob relevante Korrekturen mit den aktiven elektrischen Freiheitsgraden gekoppelt sind.

In [ ]:
g = sibc.sibc_g
active = g > 0
row_active = np.diff(sibc.C_red.tocsr().indptr) > 0
for comp, direction in enumerate("xyz"):
    m = active[comp * sibc.N : (comp + 1) * sibc.N]
    r = row_active[comp * sibc.N : (comp + 1) * sibc.N]
    print(f"{direction}: {m.sum()} SIBC faces, {(m & r).sum()} coupled")
print("Total active SIBC:", active.sum(), "coupled:", (active & row_active).sum())
print(
    "Material-only DOFs removed:",
    np.count_nonzero(
        sibc.active_conductor_E
        & sibc.bc_mask_E.toarray().astype(bool)
        & (sibc.iDeps.diagonal() == 0)
    ),
)

## 2. PEC-Resonanzfrequenz

Wir bestimmen den PEC-Eigenmode in der Nähe von $f_\mathrm{TE101}$. Für das Finite-Grid-Ergebnis ist eine gewisse Diskretisierungsabweichung normal. Falls mehrere Moden nahe beieinanderliegen, muss die TE101-Mode anhand ihrer Feldverteilung identifiziert werden.

In [ ]:
frequencies, modes, eigenvalues = pec.solve_eigenmodes(
    n_modes=6, target_frequency=f_analytical
)
print("Eigenfrequencies [GHz]:", np.round(frequencies / 1e9, 6))
valid = np.flatnonzero(np.isfinite(frequencies) & (frequencies > 0))
assert len(valid) > 0
k_mode = valid[np.argmin(np.abs(frequencies[valid] - f_analytical))]
f0 = float(frequencies[k_mode])
omega0 = 2 * pi * f0
e = modes[:, k_mode].astype(complex)
print(f"Selected mode {k_mode}: f_FIT = {f0 / 1e9:.6f} GHz")
print(f"Relative PEC frequency error = {(f0 / f_analytical - 1) * 100:.3f} %")

## 3. SIBC-Verlust und störungstheoretischer Gütefaktor

Wir benutzen die modebasierte Näherung (mit $\exp(+j\omega t)$):

$$\delta\omega \simeq \frac{\mathbf e^H[\mathbf S_{L,\mathrm{SIBC}}(\omega_0)-\mathbf S_{L,\mathrm{PEC}}]\mathbf e}{2\omega_0\,\mathbf e^H\mathbf M_\varepsilon\mathbf e},\qquad Q_{\rm FIT}\simeq\frac{\omega_0}{2\operatorname{Im}\delta\omega}. $$

Für einen passiven Leiter muss bei dieser Zeitkonvention $\operatorname{Im}\delta\omega>0$ sein. Die Näherung setzt geringe Verluste und eine ausreichend isolierte Mode voraus.

Der analytische Vergleich berücksichtigt **nur eine verlustbehaftete x-Wand**; die anderen fünf Wände bleiben ideal PEC. Mit $R_s=\sqrt{\omega\mu_0/(2\sigma)}$ gilt für TE101:

$$Q_{\mathrm{wall}} = \frac{\omega\mu_0}{R_s}\frac{a(k_x^2+k_z^2)}{2 k_x^2}. $$

Dies ist der Oberflächenverlust der analytischen Kavitätsmode, keine exakte Lösung des diskreten konformen Modells.

In [ ]:
D = sibc._build_sibc_diagonal(omega0)
mnu = sibc.Mnu.diagonal()
C = sibc.C_red.tocsr()
from scipy.sparse import diags

K_sibc = (C.T @ diags(mnu / D) @ C).tocsr()
dK = K_sibc - pec.SL
numerator = np.vdot(e, dK @ e)
denominator = 2 * omega0 * np.vdot(e, pec.Meps @ e)
domega = numerator / denominator

Rs = np.sqrt(omega0 * mu_0 / (2 * sigma_wall))
Q_analytic = (omega0 * mu_0 / Rs) * a * (kx * kx + kz * kz) / (2 * kx * kx)

print(f"R_s(PEC reference frequency): {Rs:.6g} ohm")
print(f"Complex frequency shift δω/(2π): {domega / (2 * pi):.6g} Hz")
if domega.imag > 0:
    Q_fit = omega0 / (2 * domega.imag)
    print(f"Q_FIT (first-order): {Q_fit:.4g}")
    print(f"Q_analytic (one lossy wall): {Q_analytic:.4g}")
    print(f"Ratio Q_FIT/Q_analytic: {Q_fit / Q_analytic:.4g}")
else:
    print("WARNING: no positive damping. Check SIBC sign and operator!")
print(f"Real frequency shift: {domega.real / (2 * pi):+.6g} Hz")

## 4. Leitfähigkeitsskalierung

Zur zusätzlichen Kontrolle berechnen wir $Q$ mit denselben Geometriefaktoren für mehrere Leitfähigkeiten. Erwartet wird bei gutem Leiter $Q\propto\sqrt{\sigma}$. Dieser Test prüft die Frequenz-/Leitfähigkeitsskalierung des **implementierten Operators**, noch nicht dessen absolute physikalische Richtigkeit.

In [ ]:
sigma_ref = sibc.sibc_sigma.copy()
for sigma in (1e5, 1e6, 1e7, 5.8e7):
    try:
        sibc.sibc_sigma[active] = sigma
        Dtest = sibc._build_sibc_diagonal(omega0)
        DeltaK = (C.T @ diags(mnu * (1 / Dtest - 1)) @ C).tocsr()
        delta = np.vdot(e, DeltaK @ e) / denominator
        q = omega0 / (2 * delta.imag) if delta.imag > 0 else np.nan
        print(
            f"sigma={sigma:.2e} S/m, Q={q:.6g}, Q/sqrt(sigma)={q / np.sqrt(sigma):.6g}"
        )
    finally:
        sibc.sibc_sigma[:] = sigma_ref

## Interpretation und Grenzen

- Die Geometrie enthält eine ebene Leiterfläche, wodurch unerwünschte Mehrfachschnitte deutlich reduziert werden. Melden die Geometriediagnosen dennoch ungültige Flächen, sollte die Geometrie vor weiteren Tests angepasst werden.
- **Modeidentifikation:** Der nächstliegende Eigenwert ist nicht automatisch TE101; bei unerwarteten Ergebnissen die E-Feldstruktur prüfen.
- **Vorzeichen:** $\operatorname{Im}\delta\omega>0$ ist unter der verwendeten Zeitkonvention ein notwendiger Test für positive Dämpfung.
- **Absolute Genauigkeit:** Die erste SIBC-Version nutzt die skalare $\sin\theta$-Approximation. Eine Übereinstimmung im $\sqrt\sigma$-Gesetz ist noch kein Beleg dafür, dass Geometriefaktoren und Verluste quantitativ stimmen.
- Nach dieser schnellen Störungsanalyse können wir mit einem adaptiven FD-Frequenzsweep die Resonanzbreite unabhängig bestimmen. Für Kupfer ist ein Sweep über viele Prozent der Resonanzfrequenz normalerweise zu grob, um den Gütefaktor abzulesen.
- **Snapping:** `conformal_pec_snap_fraction=0.2` applies in both material variants. The analytical frequency uses the nominal wall position; geometry snapping introduces additional discretization error.


## 5. Direct driven FD sweep around the TE101 resonance

This is an **independent test of the frequency-dependent FD system matrix**. Instead of calculating a beam-coupling impedance, we excite the system with a source proportional to the selected PEC eigenmode, then project the driven field back onto that same mode. This isolates the resonance and avoids additional trajectory/source normalization effects.

For the phasor convention $e^{+j\omega t}$, the physical damped resonance pole is at $f_p=f_r+j\gamma$, with $\gamma>0$. Fit the complex transfer function to

$$H(f)=H_{\rm bg}+\frac{B}{f-f_r-j\gamma},\qquad Q_{\rm FD}=\frac{f_r}{2\gamma}. $$

For a high-$Q$ resonator, the expected half-width is $\gamma\simeq\operatorname{Im}(\delta\omega)/(2\pi)$ and $f_r\simeq f_0+\operatorname{Re}(\delta\omega)/(2\pi)$. Here `conformal_pec_snap_fraction=0.2` remains unchanged from the earlier cells.

In [ ]:
from scipy.optimize import least_squares
import matplotlib.pyplot as plt

# Drive with the PEC eigenmode: full vector has 3*N entries, reduced source uses Meps @ e.
source = np.zeros(3 * sibc.N, dtype=complex)
source[sibc.free_E_idx] = pec.Meps @ e

f_pred = f0 + domega.real / (2 * pi)
gamma_pred = domega.imag / (2 * pi)  # half width at half maximum, Hz
assert gamma_pred > 0, "Expected positive SIBC damping; check the previous cell."

# 41 direct sparse solves cover +/- 6 predicted half-widths.
# Increase n_samples after validating the initial fit.
n_samples = 41
f_sweep = np.linspace(f_pred - 6 * gamma_pred, f_pred + 6 * gamma_pred, n_samples)
H = np.empty(n_samples, dtype=complex)

for k, f in enumerate(f_sweep):
    E_full = sibc.solve(omega=2 * pi * f, source=source)
    E_red = E_full[sibc.free_E_idx]
    H[k] = np.vdot(e, pec.Meps @ E_red) / np.vdot(e, pec.Meps @ e)

print(f"Sweep: {f_sweep[0] / 1e6:.9f} to {f_sweep[-1] / 1e6:.9f} MHz")
print(f"Predicted resonance: {f_pred / 1e6:.9f} MHz")
print(f"Predicted FWHM: {2 * gamma_pred:.3f} Hz")

### Complex resonance fit

Fit the **complex** response (not only its magnitude). This avoids bias from the complex phase and allows a constant nonresonant background. The fit is centered and scaled in kHz for numerical stability.

In [ ]:
# Use the predicted half-width to make the optimization variables dimensionless.
x = (f_sweep - f_pred) / gamma_pred
h_scale = np.max(np.abs(H))
y = H / h_scale


# y(x) = C + B / (x - x_r - 1j*g), where g > 0.
def resonance_model(p, x):
    xr, log_g, Br, Bi, Cr, Ci = p
    return (Cr + 1j * Ci) + (Br + 1j * Bi) / (x - xr - 1j * np.exp(log_g))


def residual(p):
    difference = resonance_model(p, x) - y
    return np.r_[difference.real, difference.imag]


# Approximate pole strength from peak, allowing any complex phase.
B0 = (y[np.argmin(abs(x))] - (y[0] + y[-1]) / 2) * (-1j)
C0 = (y[0] + y[-1]) / 2
p0 = [0.0, 0.0, B0.real, B0.imag, C0.real, C0.imag]
fit = least_squares(residual, p0, max_nfev=3000)
assert fit.success, fit.message
xr_fit, log_g_fit, *_ = fit.x
f_fit = f_pred + xr_fit * gamma_pred
gamma_fit = np.exp(log_g_fit) * gamma_pred
Q_direct = f_fit / (2 * gamma_fit)
H_fit = h_scale * resonance_model(fit.x, x)

print(f"f_PEC                         = {f0 / 1e6:.9f} MHz")
print(f"f_pred (first-order SIBC)     = {f_pred / 1e6:.9f} MHz")
print(f"f_fit (direct FD)             = {f_fit / 1e6:.9f} MHz")
print(f"FWHM_direct                   = {2 * gamma_fit:.3f} Hz")
print(f"Q_direct                     = {Q_direct:.2f}")
print(f"Q_first_order                = {Q_fit:.2f}")
print(f"Q_analytic                   = {Q_analytic:.2f}")
print(f"Q_direct / Q_first_order     = {Q_direct / Q_fit:.6f}")
print(f"Q_direct / Q_analytic        = {Q_direct / Q_analytic:.6f}")
print(
    f"Relative complex fit residual= {np.linalg.norm(H - H_fit) / np.linalg.norm(H):.3e}"
)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot((f_sweep - f_pred) / 1e3, abs(H), "o", label="Direct FD solves")
ax.plot((f_sweep - f_pred) / 1e3, abs(H_fit), "-", label="Complex pole fit")
ax.axvline(
    (f_fit - f_pred) / 1e3, color="gray", ls="--", lw=1, label="Fitted resonance"
)
ax.set(
    xlabel="Frequency relative to first-order prediction [kHz]",
    ylabel="|modal response| [arb. units]",
    title="Driven FD SIBC resonance",
)
ax.grid(alpha=0.3)
ax.legend()
plt.show()

### Interpretation

- Agreement between $Q_{m direct}$ and $Q_{m first\ order}$ confirms that the **driven** FD system reproduces the wall losses estimated from the SIBC perturbation.
- Agreement with $Q_{m analytic}$ checks the magnitude of wall losses against a physical reference.
- The tested quantity $H$ is a **modal transfer function, not a longitudinal beam impedance**. A beam-current source and path integral would be the next independent end-to-end validation.
- If the direct fit is poor, inspect the magnitude plot, examine the complex fit residual, and widen/refine the frequency window. A missing SIBC correction in `build_system_matrix()` will typically produce an extremely sharp/undamped PEC-like resonance and make this fit fail.
- Solving 41 sparse systems on a fine 3D mesh can be computationally expensive.